# Reproduce the v1ca1 manuscript figures

This notebook regenerates Figures 1–4 and Supplementary Figures 1–8 from the exported Spyglass database and the associated NWB files in [DANDI:001958 version 0.260829.0404](https://doi.org/10.48324/dandi.001958/0.260829.0404). Outputs are written as SVG, PDF, and 600 dpi PNG files.

In [ ]:
from importlib.metadata import version
import os
from pathlib import Path

import datajoint as dj
from dandi.dandiapi import DandiAPIClient

DANDISET_ID = "001958"
DANDISET_VERSION = "0.260829.0404"
DANDI_DOI = "https://doi.org/10.48324/dandi.001958/0.260829.0404"

_ORIGINAL_GET_DANDISET = getattr(
    DandiAPIClient,
    "_v1ca1_original_get_dandiset",
    DandiAPIClient.get_dandiset,
)
DandiAPIClient._v1ca1_original_get_dandiset = _ORIGINAL_GET_DANDISET


def _get_pinned_dandiset(
    self, dandiset_id, version_id=None, lazy=True
):
    """Resolve this export through its immutable DANDI version."""
    if str(dandiset_id) == DANDISET_ID:
        if version_id not in (None, DANDISET_VERSION):
            raise ValueError(
                f"DANDI:{DANDISET_ID} is pinned to "
                f"{DANDISET_VERSION}, not {version_id}."
            )
        version_id = DANDISET_VERSION
    return _ORIGINAL_GET_DANDISET(
        self, dandiset_id, version_id=version_id, lazy=lazy
    )


DandiAPIClient.get_dandiset = _get_pinned_dandiset
with DandiAPIClient.for_dandi_instance("dandi") as client:
    published = client.get_dandiset(
        DANDISET_ID, DANDISET_VERSION, lazy=False
    )
if published.version_id != DANDISET_VERSION:
    raise RuntimeError(
        f"Expected DANDI version {DANDISET_VERSION}; "
        f"resolved {published.version_id}."
    )

os.chdir("/home/jovyan")
NWB_ROOT = Path("/home/jovyan/data/raw")
OUTPUT_DIR = Path("/home/jovyan/notebooks/output/spyglass")
NWB_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Spyglass {version('spyglass-neuro')}")
print(f"v1ca1 {version('v1ca1')}")
print(f"DANDI:{DANDISET_ID} {DANDISET_VERSION} ({DANDI_DOI})")
dj.conn()

The default selection below reproduces the complete manuscript set. For a quick connectivity test, set `FIGURES = ("supplementary_figure_5",)` and `FORMATS = ("svg",)`.

In [ ]:
from v1ca1.paper_figures._spyglass_database import SpyglassFigureDatabase
from v1ca1.paper_figures.generate_spyglass_figures import (
    FIGURE_NAMES,
    generate_spyglass_figures,
)

FIGURES = FIGURE_NAMES
FORMATS = ("svg", "pdf", "png")
DPI = 600

database = SpyglassFigureDatabase(nwb_root=NWB_ROOT)
outputs = generate_spyglass_figures(
    database,
    output_dir=OUTPUT_DIR,
    figure_names=FIGURES,
    output_formats=FORMATS,
    dpi=DPI,
    replace=True,
)
outputs

In [ ]:
import json

manifest_path = OUTPUT_DIR / "spyglass_figure_generation.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["dandiset"] = {
    "identifier": f"DANDI:{DANDISET_ID}",
    "version": DANDISET_VERSION,
    "doi": DANDI_DOI,
}
manifest_path.write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)
{
    "outputs": len(manifest["figures"]),
    "formats": manifest["output_formats"],
    "result_rows": len(manifest["result_rows"]),
    "dandiset": manifest["dandiset"],
    "manifest": str(manifest_path),
}